In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [49]:
df = pd.read_csv('BasicSurvey(3-6-20).csv', low_memory=False)
pattern = rf"^({'usedrugs'})_[1-8]$"
tv_cols = df.columns[df.columns.str.match(pattern)].tolist()

df[tv_cols]
pattern
df.columns.str.match(pattern)

array([False, False, False, ..., False, False, False], shape=(3027,))

In [50]:
assert df["egoid"].is_unique,"Duplicate participants rows detected!"
assert df['egoid'].notna().all(), "Null primary keys detected!"


In [51]:
bases = df.columns.to_series().str.replace(r"_\d+$", "", regex=True)

schema = (
    pd.DataFrame({
        "col": df.columns,
        "base":bases.values,
        "dtype": df.dtypes.astype(str).values,
        "non_null": df.notna().sum().values
    })
    .groupby("base", as_index=False)
    .agg(
        n_waves=("col","count"),
        dtypes=("dtype", lambda s: ",".join(sorted(s.unique()))),
        total_non_null = ("non_null","sum")
    )
    .sort_values(["n_waves","total_non_null"], ascending=[False,False])
)


,base,n_waves,dtypes,total_non_null
819,physicalactivity1,8,float64,3897
722,majorevent,8,str,3088
929,usecaffine,8,str,3083
498,clubsports,8,str,3082
931,usedrugs_prescr,8,str,3082
...,...,...,...,...
665,hawaii1,1,float64,0
667,hawaii3,1,float64,0
899,studyAbroadF_6,1,float64,0
907,studyAbroad_4,1,float64,0


In [56]:
#schema.query("n_waves >= 7 & " )
#schema[schema["dtypes"].str.contains(",")]
#schema["n_waves"].value_counts().sort_index()
schema.query("n_waves >= 4" )
schema.query("n_waves >= 4" ).to_csv('categories', index=False)




In [ ]:

meta_cols = ['egoid', 'Tier', 'gender_1']
date_cols = [f"{p}_{w}" for w in range(1, 9) for p in ['StartDate', 'EndDate']]

date_cols

['StartDate_1',
 'EndDate_1',
 'StartDate_2',
 'EndDate_2',
 'StartDate_3',
 'EndDate_3',
 'StartDate_4',
 'EndDate_4',
 'StartDate_5',
 'EndDate_5',
 'StartDate_6',
 'EndDate_6',
 'StartDate_7',
 'EndDate_7',
 'StartDate_8',
 'EndDate_8']

In [ ]:


tv_bases = [
    'Weight', 'Height',
    'usetobacco', 'usecaffine', 'usebeer', 'usewine', 'usedrugs', 'usedrugs_prescr'
]
tv_cols = [f"{b}_{w}" for b in tv_bases for w in range(1, 9) if f"{b}_{w}" in df.columns]

selected_cols = meta_cols + date_cols  + tv_cols
assert len(selected_cols) == len(set(selected_cols)), "Duplicate columns!"
assert all(c in df.columns for c in selected_cols), "Missing columns!"

df_clean = df[selected_cols].copy()
#out_path = '/mnt/data/BasicSurvey_cleaned_139cols.csv'
#df_clean.to_csv(out_path, index=False)

#print("Saved:", out_path)
print("Shape:", df_clean.shape)
print("\nSample unique values for ordinal/binary columns (Wave 1 or 2):")
for col in [ 'usetobacco_1', 'usecaffine_1', 'usebeer_1',
            'usewine_1', 'usedrugs_1', 'usedrugs_prescr_1']:
    print(f"  {col:20s}: {df_clean[col].dropna().unique().tolist()}")

Shape: (722, 82)

Sample unique values for ordinal/binary columns (Wave 1 or 2):
  usetobacco_1        : ['Not at all', '1-2 times a week', 'Less than 1-2 times a month', '1-2 times a month', 'Three times a week or more']
  usecaffine_1        : ['1-2 times a month', '1-2 times a week', 'Three times a week or more', 'Less than 1-2 times a month', 'Not at all']
  usebeer_1           : ['Not at all', 'Less than 1-2 times a month', '1-2 times a month', '1-2 times a week', 'Three times a week or more']
  usewine_1           : ['Not at all', '1-2 times a week', 'Less than 1-2 times a month', '1-2 times a month', 'Three times a week or more']
  usedrugs_1          : ['Not at all', '1-2 times a month', 'Less than 1-2 times a month', '1-2 times a week', 'Three times a week or more']
  usedrugs_prescr_1   : ['Not at all', 'Less than 1-2 times a month', 'Three times a week or more']


In [32]:
expected_vocab = {
    'Not at all', 
    '1-2 times a week', 
    'Less than 1-2 times a month', 
    '1-2 times a month', 
    'Three times a week or more'
}

category_bases = ['usetobacco', 'usecaffine', 'usebeer', 'usewine', 'usedrugs', 'usedrugs_prescr']

for base in category_bases:
    cols = [f"{base}_{w}" for b in base for w in range(1,9)]
    observed = set(pd.Series(df_clean[cols].values.ravel()).dropna())                                            #set only keeps unique values
    extra = observed - expected_vocab
    assert not extra, f"Unexpected lables in {base}: {extra}"

In [28]:
df_clean[cols].values.ravel()

array(['Not at all', 'Not at all', 'Not at all', ..., 'Not at all',
       'Not at all', nan], shape=(86640,), dtype=object)

In [ ]:



df = pd.read_csv("BasicSurvey_cleaned_139cols.csv")

# 1. Fill missing Height within each person (across waves 1..8) and compute BMI (lbs/in^2 * 703)
h_cols = [f"Height_{w}" for w in range(1, 9) if f"Height_{w}" in df.columns]
height_filled = df[h_cols].ffill(axis=1).bfill(axis=1).median(axis=1)

for w in range(1, 9):
    df[f"BMI_{w}"] = (df[f"Weight_{w}"] / (height_filled ** 2)) * 703

# 2. Map ordinal frequency strings to numeric levels (including '0' in exercise_1)
freq_map = {
    "0": 0,
    "Not at all": 0,
    "Less than 1-2 times a month": 1,
    "1-2 times a month": 2,
    "1-2 times a week": 3,
    "Three times a week or more": 4,
}

# 4. Merge Alcohol (beer + wine) and Drugs (illicit + prescription)
for w in range(1, 9):
    df[f"use_alcohol_{w}"] = np.fmax(df[f"usebeer_{w}"].map(freq_map), df[f"usewine_{w}"].map(freq_map))
    df[f"use_drugs_{w}"] = np.fmax(df[f"usedrugs_{w}"].map(freq_map), df[f"usedrugs_prescr_{w}"].map(freq_map))